# Member 1: Synthetic data, preprocessing and state-action encoding

This notebook calls the same implementation as the CLI. It does not extract a real Sepsis-3 cohort. Run this notebook before Member 2 and Member 3.

In [ ]:
import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import Image, display

root = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'main_pipeline.py').is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project root or a member folder")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from main_pipeline import load_configs, prepare_data, train_model, evaluate_model

output = Path(os.environ.get('ASS2_OUTPUT_DIR', root)).resolve()
print("Source: synthetic coursework demo; no clinical efficacy claim.")
print("Output:", output)


## 1. Prepare the reproducible cohort

Existing outputs are reused only when configuration matches. To regenerate, use the CLI with an explicit output directory and `--overwrite`. Current treatment effects enter the next state, not the current observation.

In [ ]:
configs = load_configs()
manifest_path = output / 'artifacts/run_config.json'
if (output / 'data/raw_sepsis_cohort.csv').exists():
    if not manifest_path.exists() or json.loads(manifest_path.read_text(encoding='utf-8')) != configs:
        raise RuntimeError("Existing run has a different or missing configuration; select a new output directory")
    print("Reusing prepared synthetic data.")
else:
    prepare_data(output, configs)
raw = pd.read_csv(output / 'data/raw_sepsis_cohort.csv')
print("Observations:", len(raw), "| Patients:", raw['subject_id'].nunique())
raw.head()


## 2. Patient-level splits and episode boundaries

All stays belonging to a patient remain in one split; each ICU stay is a separate episode.

In [ ]:
trajectories = {name: pd.read_csv(output / f'data/trajectories_{name}.csv') for name in ['train', 'val', 'test']}
ids = {name: set(frame['subject_id']) for name, frame in trajectories.items()}
assert not (ids['train'] & ids['val'] or ids['train'] & ids['test'] or ids['val'] & ids['test'])
pd.DataFrame({name: {'patients': len(ids[name]), 'episodes': int(frame['done'].sum()), 'steps': len(frame)} for name, frame in trajectories.items()}).T


## 3. Train-fitted preprocessing and encoding

Training medians fill missing values; the StandardScaler and K-means centroids fit training observations only. Positive dose quartiles also come from training.

In [ ]:
preprocessor = joblib.load(output / 'artifacts/preprocessor.joblib')
encoder = joblib.load(output / 'artifacts/state_action_encoder.joblib')
print("Feature count:", len(preprocessor.feature_cols))
print("Health states:", encoder.n_states, "| Terminal states:", encoder.alive_state, encoder.dead_state)
print("Fluid cutoffs:", encoder.fluid_bins)
print("Vasopressor cutoffs:", encoder.vaso_bins)
pd.Series(preprocessor.medians, name='training_median')


## 4. Inspect the trajectory contract

Intermediate rewards are zero; the final transition receives +100 or -100 from a simulated outcome. These are not measured 90-day outcomes.

In [ ]:
first_subject = trajectories['train']['subject_id'].iloc[0]
trajectories['train'].query('subject_id == @first_subject')


## 5. Audit and provenance

The audit reports checks it actually performed. The raw-data report cannot independently prove model fit provenance.

In [ ]:
display(json.loads((output / 'artifacts/cohort_manifest.json').read_text()))
from IPython.display import Markdown
display(Markdown((output / 'docs/data_quality_report.md').read_text(encoding='utf-8')))
